# module-composition — ex1: child Modules auto-register as attributes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-composition`. Running the final beacon cell reports progress against the `PyTorch: Module composition` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Module composition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-composition`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-composition"
DD_SUBTOPIC = "PyTorch: Module composition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Module composition — quick refresher

Modules compose by assignment. Any `nn.Module` instance assigned to `self.<name>` inside `__init__` is auto-registered as a child module — visible in `.children()`, `.named_modules()`, and recursively included in `.parameters()` / `.state_dict()`.

**Two composition styles.**
1. **Named attributes** — `self.linear1 = nn.Linear(...)`, `self.linear2 = nn.Linear(...)`. Use when the forward pass branches (residual blocks, attention heads, gating).
2. **`nn.Sequential(*modules)`** — wraps a list of Modules into a single callable that pipes the output of one into the input of the next. Use when the forward is a strict left-to-right pipeline.

**Lists need `nn.ModuleList`, not `list`.** A plain Python list of Modules assigned to an attribute does NOT register the children — their parameters become invisible. Use `nn.ModuleList(...)` (registers + supports indexing) or `nn.Sequential(...)` (registers + auto-pipes).

### Exercise 1 — child Modules auto-register as attributes

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Compose a parent Module from two named child Modules via attribute assignment, and verify the parent's parameters() transitively includes the children's parameters.
> Keywords: children, named_modules, composition, attribute-registration
> ```

**KCs targeted:** `child-module-attribute-registration`

Implement `TwoLayerMLP` — the simplest possible module-composition pattern. Subclass `t.nn.Module` and in `__init__(self, in_features, hidden_features, out_features)`:

1. Call `super().__init__()` first.
2. Assign two child Modules as named attributes:
   - `self.fc1 = t.nn.Linear(in_features, hidden_features)`
   - `self.fc2 = t.nn.Linear(hidden_features, out_features)`
3. `forward(self, x: Tensor) -> Tensor` computes `fc2(relu(fc1(x)))` using `t.relu` between the two linears.

Return an instance from `ex1_build_two_layer_mlp(in_features, hidden_features, out_features)`.

The test confirms: (a) both child Linears show up in `.children()` and `.named_modules()`, (b) the parent's `.parameters()` recursively contains all 4 tensors (fc1.weight, fc1.bias, fc2.weight, fc2.bias), (c) the forward pass produces the correct shape, (d) the children are auto-named after the attributes you assigned them to.

In [ ]:
def ex1_build_two_layer_mlp(in_features: int, hidden_features: int, out_features: int):
    class TwoLayerMLP(t.nn.Module):
        def __init__(self, in_features, hidden_features, out_features):
            super().__init__()
            self.fc1 = t.nn.Linear(in_features, hidden_features)
            self.fc2 = t.nn.Linear(hidden_features, out_features)
        def forward(self, x: Tensor) -> Tensor:
            return self.fc2(t.relu(self.fc1(x)))
    return TwoLayerMLP(in_features, hidden_features, out_features)


<details><summary>Solution</summary>

```python
def ex1_build_two_layer_mlp(in_features: int, hidden_features: int, out_features: int):
    class TwoLayerMLP(t.nn.Module):
        def __init__(self, in_features, hidden_features, out_features):
            super().__init__()
            self.fc1 = t.nn.Linear(in_features, hidden_features)
            self.fc2 = t.nn.Linear(hidden_features, out_features)
        def forward(self, x: Tensor) -> Tensor:
            return self.fc2(t.relu(self.fc1(x)))
    return TwoLayerMLP(in_features, hidden_features, out_features)
```

**The auto-registration mechanism.** `nn.Module.__setattr__` checks the value of every attribute assignment. If it's an `nn.Module`, it goes into `self._modules['fc1']` (named after the attribute). That's why `named_children()` returns the names you used in your code, and `named_parameters()` returns dot-prefixed paths like `fc1.weight`.

**The forgotten-relu bug.** A common slip is `self.fc2(self.fc1(x))` — no nonlinearity between linears, making the whole stack equivalent to a single linear transformation. The test's negative-fc1 trick catches exactly this: with all-negative fc1 outputs, ReLU should zero them and fc2 produces all-bias; without ReLU, fc2 would produce a non-zero value.

**Why `nn.ModuleList` not `list`.** If you'd written `self.layers = [Linear(...), Linear(...)]`, the children wouldn't register — `mod.parameters()` would be empty and training silently wouldn't update the layers.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()